In [1]:
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# statistics (optional but useful)
import scipy as sp
from scipy import stats

In [2]:
import chardet

with open('data_source/bikeshare-ridership-2025/bikeshare_2025_01.csv', 'rb') as f:
    result = chardet.detect(f.read(100000))
    print(result)

{'encoding': 'Windows-1252', 'confidence': 0.73, 'language': ''}


In [3]:
df_jan_25 = pd.read_csv('data_source/bikeshare-ridership-2025/bikeshare_2025_01.csv', 
                          encoding='cp1252')
df_jan_25.head()
df_jan_25.shape
df_jan_25.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 202946 entries, 0 to 202945
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Trip_Id             202946 non-null  int64  
 1   Trip_Duration       202946 non-null  int64  
 2   Start_Station_Id    202946 non-null  int64  
 3   Start_Time          202946 non-null  object 
 4   Start_Station_Name  202946 non-null  object 
 5   End_Station_Id      202811 non-null  float64
 6   End_Time            202920 non-null  object 
 7   End_Station_Name    202946 non-null  object 
 8   Bike_Id             202946 non-null  int64  
 9   User_Type           202946 non-null  object 
 10  Bike_Model          202946 non-null  object 
dtypes: float64(1), int64(4), object(6)
memory usage: 17.0+ MB


In [4]:
df_jan_25['User_Type'].unique()


array(['Member', 'Casual'], dtype=object)

In [5]:
df_jan_25['Bike_Model'].unique()

array(['ICONIC', 'EFIT G5', 'EFIT'], dtype=object)

In [6]:
import glob

path = 'data_source/bikeshare-ridership-2025/'
files = sorted(glob.glob(path + 'bikeshare_2025_*.csv'))

print(len(files))  # confirm 12 months found

12


In [7]:
for f in files:
    temp = pd.read_csv(f, encoding='cp1252', nrows=5)  # just read 5 rows, fast check
    print(f, '->', list(temp.columns))

data_source/bikeshare-ridership-2025/bikeshare_2025_01.csv -> ['Trip_Id', 'Trip_Duration', 'Start_Station_Id', 'Start_Time', 'Start_Station_Name', 'End_Station_Id', 'End_Time', 'End_Station_Name', 'Bike_Id', 'User_Type', 'Bike_Model']
data_source/bikeshare-ridership-2025/bikeshare_2025_02.csv -> ['Trip_Id', 'Trip_Duration', 'Start_Station_Id', 'Start_Time', 'Start_Station_Name', 'End_Station_Id', 'End_Time', 'End_Station_Name', 'Bike_Id', 'User_Type', 'Bike_Model']
data_source/bikeshare-ridership-2025/bikeshare_2025_03.csv -> ['Trip_Id', 'Trip_Duration', 'Start_Station_Id', 'Start_Time', 'Start_Station_Name', 'End_Station_Id', 'End_Time', 'End_Station_Name', 'Bike_Id', 'User_Type', 'Bike_Model']
data_source/bikeshare-ridership-2025/bikeshare_2025_04.csv -> ['Trip_Id', 'Trip_Duration', 'Start_Station_Id', 'Start_Time', 'Start_Station_Name', 'End_Station_Id', 'End_Time', 'End_Station_Name', 'Bike_Id', 'User_Type', 'Bike_Model']
data_source/bikeshare-ridership-2025/bikeshare_2025_05.csv -

In [8]:
df_2025 = [pd.read_csv(f, encoding='cp1252') for f in files]
df_25 = pd.concat(df_2025, ignore_index=True)

print(df_25.shape)
df_25.info()

(7812520, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7812520 entries, 0 to 7812519
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Trip_Id             int64  
 1   Trip_Duration       int64  
 2   Start_Station_Id    float64
 3   Start_Time          object 
 4   Start_Station_Name  object 
 5   End_Station_Id      float64
 6   End_Time            object 
 7   End_Station_Name    object 
 8   Bike_Id             int64  
 9   User_Type           object 
 10  Bike_Model          object 
dtypes: float64(2), int64(3), object(6)
memory usage: 655.7+ MB


In [9]:
df_25['Start_Station_Id'].isna().sum()

7

In [10]:
df_25['End_Station_Id'].isna().sum()

6761

In [11]:
df_25['End_Time'].isna().sum()

1092

In [12]:
df_25.isna().sum()


Trip_Id                  0
Trip_Duration            0
Start_Station_Id         7
Start_Time               0
Start_Station_Name       7
End_Station_Id        6761
End_Time              1092
End_Station_Name       924
Bike_Id                  0
User_Type                0
Bike_Model               0
dtype: int64

In [13]:
df_25.duplicated().sum()

0

In [14]:
df_25['Trip_Id'].duplicated().sum()  # Trip_Id should be unique

0

In [15]:
df_25[df_25['Start_Station_Id'].isna()]

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
5409066,40954646,40,NaN,2025-09-08 13:46:28,NaN,NaN,2025-09-08 13:47:08,NaN,10618,Member,ICONIC
5409646,40956135,11,NaN,2025-09-08 14:37:50,NaN,NaN,2025-09-08 14:38:01,NaN,10618,Member,ICONIC
5409997,40956146,5,NaN,2025-09-08 14:38:09,NaN,NaN,2025-09-08 14:38:14,NaN,10618,Member,ICONIC
5410513,40956155,77,NaN,2025-09-08 14:38:23,NaN,NaN,2025-09-08 14:39:40,NaN,10618,Member,ICONIC
5507074,41065430,17,NaN,2025-09-10 22:43:28,NaN,NaN,2025-09-10 22:43:45,NaN,8534,Member,EFIT
5507462,41065464,6,NaN,2025-09-10 22:44:52,NaN,NaN,2025-09-10 22:44:58,NaN,8534,Member,EFIT
5924410,41541544,14,NaN,2025-09-22 16:32:42,NaN,NaN,2025-09-22 16:32:56,NaN,10618,Member,ICONIC


In [16]:
df_25[df_25['End_Station_Id'].isna()].head(10)


,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
190,34637838,1138,7432.0,2025-01-01 00:47:02,Frederick St / King St E,NaN,2025-01-01 01:06:00,Frederick St / King St E,212,Casual,ICONIC
692,34638383,0,7770.0,2025-01-01 03:47:06,Spadina Ave / Sullivan St,NaN,NaN,Spadina Ave / Sullivan St,2212,Casual,ICONIC
1325,34639326,0,7660.0,2025-01-01 12:13:38,285 Victoria St,NaN,2025-01-01 12:13:38,285 Victoria St,2249,Member,ICONIC
2264,34640716,0,7043.0,2025-01-01 16:13:01,Queens Quay W / Lower Simcoe St,NaN,NaN,Queens Quay W / Lower Simcoe St,646,Casual,ICONIC
2914,34641136,0,7416.0,2025-01-01 17:14:21,Spadina Ave / Blue Jays Way,NaN,2025-01-01 17:14:21,Spadina Ave / Blue Jays Way,4764,Casual,ICONIC
3279,34641638,0,7164.0,2025-01-01 18:39:58,Gould St / Yonge St (TMU),NaN,NaN,Gould St / Yonge St (TMU),189,Member,ICONIC
3828,34642124,0,7037.0,2025-01-01 20:46:28,Bathurst St / Dundas St W,NaN,NaN,Bathurst St / Dundas St W,4157,Member,ICONIC
6465,34645747,0,7239.0,2025-01-02 12:42:49,Bloor St W / Manning Ave - SMART,NaN,2025-01-02 12:42:49,Bloor St W / Manning Ave - SMART,795,Member,ICONIC
10945,34650796,0,7272.0,2025-01-03 00:34:09,Yonge St / Dundonald St,NaN,2025-01-03 00:34:09,Yonge St / Dundonald St,407,Member,ICONIC
15280,34655689,0,7235.0,2025-01-03 16:09:32,Bay St / College St (West Side) - SMART,NaN,2025-01-03 16:09:32,Bay St / College St (West Side) - SMART,7322,Member,ICONIC


In [17]:
# check trip duration distribution for rows with null End_Station_Id
df_25[df_25['End_Station_Id'].isna()]['Trip_Duration'].describe()

count      6761.000000
mean       1855.545777
std        8536.544932
min           0.000000
25%           0.000000
50%         351.000000
75%        1142.000000
max      151037.000000
Name: Trip_Duration, dtype: float64

In [18]:
df_25[df_25['End_Station_Id'].isna()]['Bike_Model'].value_counts()

Bike_Model
ICONIC     5432
EFIT        957
EFIT G5     372
Name: count, dtype: int64

In [19]:
null_rows = df_25[df_25['End_Station_Id'].isna()]
(null_rows['Start_Station_Name'] == null_rows['End_Station_Name']).value_counts()

True     5837
False     924
Name: count, dtype: int64